# <font color="#2E86C1">UAT Issue:</font>

* <font color="red">**Often drift outside the intended product type or gender.**</font>

* <font color="red">**Relative Comparison**</font>
* <font color="red">**Age Context**</font>
* <font color="red">**Activity/Use Enrichment**</font>

# MAJOR PROMPT CHANGES
### ⚠️ A DIVIDED prompt into a SYSTEM prompt and a USER template.
* **✅ SYSTEM PROMPT (everything “non-negotiable”, authoritative)**
* **the model treats system messages as higher priority than user messages.**

**✅ USER TEMPLATE (data only)**  
**Generate a normalized semantic summary using the rules already provided.**

**Product Title: {Product_Title}**  
**Gender by Age: {Gender_by_Age}**  
**Brand: {Brand}**  
**Product Type: {Product_Type}**  
**Product Activities: {Product_Activities}**  
**Product Description: {Product_Description}**

### ⚠️ B. 80–150 words is too long for e5-small-v2
**e5-small-v2 performs best when:**

* **Core semantics are front-loaded**

* **Input length ~50–90 tokens (not words)**

* **80–150 words → ~120–220 tokens → increases dilution and noise.**

### 🔹 Better: 45–90 words total, 3–4 sentences.

### ⚠️ C. Brand is be de-emphasized, not primary
**Brand is a weak semantic discriminator and can distort retrieval.**

**🔹 Better hierarchy:**

* **Product_Type**

* **Physical/material properties**

* **Functional structure**

* **Gender_by_Age**

* **Brand (last)**

### ⚠️ D.Ban Marketing Phrases
**PROHIBITED PHRASES (DO NOT USE):**
- "designed for [Product_Activities]"
- "features include" / "features an"
- "perfect for" / "great way to"
- "making it easy to"
- "in a comfortable and stylish way"
- "as an officially licensed product"
- "They are"
- "This product"
- "Officially licensed product"

### <span style="color:green">**Neighbor purity**</span>

<div style="font-size:1.2em;">

**How strong is 0.9167?** 
**Last time: 0.82**

**With THIS setup (k=6 → 5 neighbors)**

* 🟢 **91.7% of nearest neighbors share the same label**
* **That indicates very strong local coherence**
* **This is well above what we usually see unless:**
    * Labels are clean
    * Embeddings are well-aligned
    * Or classes are trivially separable
* **Embeddings preserve local semantic structure**
* **Similar items cluster tightly**
* **k-NN style retrieval is likely to be stable**
* **Recent changes (model / prompt / centering) are working**

</div>

# OLD PROMPT EMBEDDINGS VISUALIZATION (PCA)

![old pca graph](./pca_plot.png)

![New Prompt PCA Plot](./pca_plot_new_prompt.png)

### Top 5 explained variance: [0.23741823 0.16103643 0.09240574 0.08061212 0.04367591]  
### Cumulative top 5: 0.6151484

### This Graph is search on "feather coat" K=48

![umap](./UMAP_of_product_embeddings.png)

# NEXT STEPS

* **size-to-age mapping (e.g., “Designed for ages 8–10”)**
* **Tune kNN parameters: sweep k and num_candidates (and HNSW params if applicable) to find the best quality/latency point.**
* **Archtecture for exception cases** 
* **New prompt Review** 
* **Code Review and checkin if UAT passed**
* **Offline relevance evaluation**

In [1]:
from elasticsearch import Elasticsearch, helpers
import os
import numpy as np
import umap
import json
import matplotlib.pyplot as plt
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
env = "DEV"
MODEL_ID = "intfloat__e5-small-v2"
MODEL_NAME = "e5"
#INDEX_RAW = "pp_vs_raw_docs-v1"        # stores original raw documents
#INDEX_VECTOR = "pp-vs-e5-llmd-embeddings-hnsw"  # stores documents with vector embeddings
INDEX_NAME_RAW = "pp-vs-e5-llmd-embeddings-newprompt-uat-raw"
PIPELINE_ID = f"{MODEL_NAME.lower()}_embedding_base_pipeline"
SHOULD_DELETE_INDEX = False
#CLOUD_ID="dsg-search-dev-east:ZWFzdHVzLmF6dXJlLmVsYXN0aWMtY2xvdWQuY29tOjQ0MyQ1Zjc1NDk5MTVmNmE0ZTBjYWFhYzYzM2IwZDI1MjNlZSRiZTg1OWU2ZGM0ZTg0Y2RmYTVjNDNkNzJiZDA3ZTg3Mg=="
ES_HOST = os.getenv("ELASTIC_HOST_DEV")
ES_USER = os.getenv("ELASTIC_USERNAME")
ES_PASS = os.getenv("ELASTIC_PASSWORD")
dims = 384 

In [3]:
MODEL_ID_MAP = {
    "e5": "intfloat__e5-small-v2",
    "sbert": "sentence-transformers__all-MiniLM-L6-v2",
    "elser": ".elser_model_2"
}

In [4]:
def get_embedding_column(model_name: str) -> str:
    model_name = model_name.lower()
    if model_name == "sbert":
        return "vs_sbert_llm_product_desc_embedding"
    elif model_name == "e5":
        return "vs_e5_llm_product_desc_embedding"
    else:
        return "content_embedding"

In [5]:
es = Elasticsearch(
    ES_HOST,  # Replace with your actual host
    basic_auth=(ES_USER,ES_PASS),
    request_timeout=600,
    verify_certs=True  # Optional: Set to False only if using self-signed certs
)

# Test connection
print(es.info().body)

{'name': 'instance-0000000096', 'cluster_name': '5f7549915f6a4e0caaac633b0d2523ee', 'cluster_uuid': 'hrds7YuPRLy60DbQMfN9Ow', 'version': {'number': '8.19.3', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '1fde05a4d63448377eceb8fd3d51ce16ca3f02a9', 'build_date': '2025-08-26T02:35:34.366492370Z', 'build_snapshot': False, 'lucene_version': '9.12.2', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [19]:
def run_query(es: Elasticsearch, query_text: str, size: int): 
    """Run a KNN query against the index"""
    search_field = get_embedding_column(MODEL_NAME)
    query_text = "query: " + query_text
    print(search_field)
    print("size=",size) 
    print(query_text)
    knn_query_body = { 
        "size" : size,
        "knn": {
            "field": search_field,
            "k": size,  # Number of nearest neighbors to retrieve
            "num_candidates": max(50 * size, 1000),# Number of candidates to consider during search
            "query_vector_builder": {
                "text_embedding": {
                    "model_id": MODEL_ID,    # deployed SBERT/E5 model
                    "model_text": query_text      # the query text to embed
                }
            }   
        }   
    }   

    response = es.search(
        index=INDEX_NAME_RAW,
        body=knn_query_body
    )   

    results = [ 
        {   
            "score": round(hit["_score"],3),
            "title": hit["_source"].get("llm_description", "")[8:],
            "id": hit.get("_id", "N/A")
        }   
        for hit in response["hits"]["hits"]
    ]   
    print(json.dumps(results, indent=2))

In [28]:
run_query(es, "womens basketball shirts", 48)

vs_e5_llm_product_desc_embedding
size= 48
query: womens basketball shirts
[
  {
    "score": 0.953,
    "title": " Womens basketball apparel, cotton short-sleeve shirt. Standard fit, crewneck, printed graphics with interlocking Cs logo. Soft cotton fabric, short sleeves. Basketball activity context.",
    "id": "25NIKWRUNNCTLNLGTXRLY"
  },
  {
    "score": 0.951,
    "title": " Womens athletic shirt, long-sleeve graphic basketball tee. Soft cotton fabric, roomy fit. Printed graphics on front and back, embroidered Swoosh on left chest, embroidered Swoosh and basketball on left sleeve. Basketball activity context.",
    "id": "24NIKWCBSKTBLLLSQAPT"
  },
  {
    "score": 0.951,
    "title": " Womens basketball shirt, soft cotton fabric, loose fit, crewneck, longer sleeves, dropped shoulders. Printed graphic with interlocking Cs logo. Nike branding, casual and basketball activity context.",
    "id": "25NIKWCASUCTLNLGGRBEE"
  },
  {
    "score": 0.947,
    "title": " Womens athletic shirt,

In [8]:
def compute_global_mean(es, index_name, source_field):
    print("Computing μ in streaming mode...")
    count = 0
    mu = None

    for doc in helpers.scan(
        es,
        index=index_name,
        query={"query": {"match_all": {}}, "_source": [source_field]},
        scroll="5m",
        size=500
    ):
        v = doc["_source"].get(source_field)
        if v:
            v = np.array(v, dtype=np.float32)
            if mu is None:
                mu = np.zeros_like(v)
            mu += v
            count += 1

    if count == 0:
        raise RuntimeError(f"No embeddings found in {index_name} field={source_field}")
    mu /= count

    print(f"Computed μ from {count} embeddings.")
    return mu.tolist()

In [9]:
def center_and_normalize(v, mu_np: np.ndarray) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32)      # query vector list -> float32 ndarray
    v = v - mu_np                            # subtract mean
    denom = np.linalg.norm(v)
    if denom == 0 or not np.isfinite(denom):
        return v  # or return original v; edge case
    return v / denom


In [10]:
def embed_query_centered(es, text: str, model_id: str, mu_np: np.ndarray) -> np.ndarray:
    resp = es.ml.infer_trained_model(
        model_id=model_id,
        docs=[{"text_field": f"query: {text}"}]
    )
    v = resp["inference_results"][0]["predicted_value"]
    return center_and_normalize(v, mu_np)


In [11]:
def knn_search_centered(es, index, field, query_vec, k=16):
    body = {
        "size": k,
        "knn": {
            "field": field,
            "k": k,
            "num_candidates": max(1000, 50 * k),
            "query_vector": query_vec.tolist()
        }
    }

    return es.search(index=index, body=body)

In [12]:
#compute μ from RAW_INDEX   
#mu_list = compute_global_mean(es, INDEX_NAME_RAW, get_embedding_column(MODEL_NAME))



In [13]:
# Convert once for client-side math
#mu_np = np.array(mu_list, dtype=np.float32)
#test query only
#query_text = "comfortable running shoes"
#q_vec = embed_query_centered(es, query_text, MODEL_ID, mu_np)
#resp = knn_search_centered(es,INDEX_VECTOR,get_embedding_column(MODEL_NAME),q_vec,k=48)

#for h in resp["hits"]["hits"]:
#    print(h["_score"], h["_id"], h["_source"]['llm_description'])


In [14]:
# 
import gradio as gr
import json
from typing import Any, Dict, List

# -------------------------------------------------------------------
# Plug your existing Elasticsearch client + search function here
# -------------------------------------------------------------------
def run_query( query_text: str, size=48): 
    """Run a KNN query against the index"""
    search_field = get_embedding_column(MODEL_NAME)
    query_text = "query: " + query_text
    print(search_field)
    print("size=",size) 
    print(query_text)
    knn_query_body = { 
        "size" : size,
        "knn": {
            "field": search_field,
            "k": size,  # Number of nearest neighbors to retrieve
            "num_candidates": max(50 * size, 1000),# Number of candidates to consider during search
            "query_vector_builder": {
                "text_embedding": {
                    "model_id": MODEL_ID,    # deployed SBERT/E5 model
                    "model_text": query_text      # the query text to embed
                }
            }   
        }   
    }   

    response = es.search(
        index=INDEX_NAME_RAW,
        body=knn_query_body
    )   

    results = [ 
        {   
            "score": hit["_score"],
            "title": hit["_source"].get("llm_description", "N/A"),
            "id": hit.get("_id", "N/A")
        }   
        for hit in response["hits"]["hits"]
    ]   
    return response

# -------------------------------------------------------------------
# Parse ES hits -> display rows
# -------------------------------------------------------------------
def parse_es_results(resp: Dict[str, Any], max_chars: int = 280) -> List[Dict[str, Any]]:
    hits = (resp or {}).get("hits", {}).get("hits", []) or []
    out: List[Dict[str, Any]] = []

    for h in hits:
        src = h.get("_source", {}) or {}
        llm_desc = src.get("llm_description", "") or ""
        llm_desc = llm_desc.replace("\n", " ").strip()
        if len(llm_desc) > max_chars:
            llm_desc = llm_desc[:max_chars] + "..."

        out.append({
            "score": h.get("_score"),
            "ecode": h.get("_id"),
            "llm_description": llm_desc[8:],
        })

    return out


# -------------------------------------------------------------------
# Gradio handler
# -------------------------------------------------------------------
def ui_search(query_text: str) -> str:
    query_text = (query_text or "").strip()
    if not query_text:
        return "<div style='color:#b00020;'>Please enter a search term.</div>"

    try:
        resp = run_query(query_text)
        rows = parse_es_results(resp)

        if not rows:
            return "<div>No results.</div>"

        # Build a scrollable HTML panel
        cards = []
        for r in rows:
            cards.append(f"""
            <div style="
                border:1px solid #e5e7eb;
                border-radius:12px;
                padding:12px 14px;
                margin-bottom:10px;
                background:#ffffff;
            ">
            <div style="margin-top:6px; font-size:13px;">
  <span style="font-weight:600; color:#6B7280;">ecode:</span>
  <span
    style="
      margin-left:4px;
      font-family:ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas, 'Liberation Mono', 'Courier New', monospace;
      color:#000000;
      font-weight:700;
    "
  >
    {r.get("ecode", "") or ""}
  </span>
</div>

              <div style="margin-top:8px; font-size:13px; color:#374151; line-height:1.35;">
                {r.get("llm_description","") or ""}
              </div>
            </div>
            """)

        html = f"""
        <div style="
            max-height:520px;
            overflow-y:auto;
            padding-right:6px;
        ">
          {''.join(cards)}
        </div>
        """
        return html

    except Exception as e:
        return f"<div style='color:#b00020;'>Error: {str(e)}</div>"


# -------------------------------------------------------------------
# Gradio UI
# -------------------------------------------------------------------
CSS = """
#app_title {
  font-weight: 800;
  color: #0a8f3c; /* green */
  font-size: 26px;
  margin: 6px 0 14px 0;
}
"""

with gr.Blocks(css=CSS) as demo:
    gr.HTML("<div id='app_title'>Dick's Vector Search Application</div>")

    with gr.Row():
        query_in = gr.Textbox(
            label="Search",
            placeholder="e.g., feather coat, garden shoes, down jacket...",
            lines=1
        )
        search_btn = gr.Button("Search", variant="primary")

    results_html = gr.HTML(value="<div>Enter a query and click Search.</div>")

    # click + enter key
    search_btn.click(fn=ui_search, inputs=query_in, outputs=results_html)
    query_in.submit(fn=ui_search, inputs=query_in, outputs=results_html)

#demo.launch()
demo.launch(server_name="0.0.0.0", server_port=7861)

/var/folders/36/3pkr81w50c72tdrmbywpc2540000gp/T/ipykernel_20628/2345240727.py:142: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS) as demo:


* Running on local URL:  http://0.0.0.0:7861
* To create a public link, set `share=True` in `launch()`.


vs_e5_llm_product_desc_embedding
size= 48
query: feather coat
vs_e5_llm_product_desc_embedding
size= 48
query: feather coat


In [16]:
import gradio as gr
import json
import os
from datetime import datetime
from typing import Any, Dict, List

# -------------------------------------------------------------------
# Logging Logic
# -------------------------------------------------------------------
def log_disliked_item(data_json: str):
    """Appends record details to a log file when 'x' is clicked."""
    if not data_json:
        return
    try:
        data = json.loads(data_json)
        data["timestamp"] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        with open("disliked_records.log", "a", encoding="utf-8") as f:
            f.write(json.dumps(data) + "\n")
        print(f"Logged dislike for ecode: {data.get('ecode')}")
    except Exception as e:
        print(f"Logging error: {e}")

# -------------------------------------------------------------------
# Modified Parse Function (Added Product Type, Gender, Title)
# -------------------------------------------------------------------
def parse_es_results(resp: Dict[str, Any], max_chars: int = 280) -> List[Dict[str, Any]]:
    hits = (resp or {}).get("hits", {}).get("hits", []) or []
    out: List[Dict[str, Any]] = []

    for h in hits:
        src = h.get("_source", {}) or {}
        llm_desc = src.get("llm_description", "") or ""
        llm_desc = llm_desc.replace("\n", " ").strip()
        if len(llm_desc) > max_chars:
            llm_desc = llm_desc[:max_chars] + "..."

        out.append({
            "score": h.get("_score"),
            "ecode": h.get("_id"),
            "llm_description": llm_desc[8:] if len(llm_desc) > 8 else llm_desc,
            "product_type": src.get("product_type", "N/A"),
            "gender": src.get("gender", "N/A"),
            "title": src.get("title", "N/A") # Assuming 'title' exists in your source
        })
    return out

# -------------------------------------------------------------------
# Modified UI Search (Added Checkbox/Log logic and Darker Borders)
# -------------------------------------------------------------------
def ui_search(query_text: str) -> str:
    query_text = (query_text or "").strip()
    if not query_text:
        return "<div style='color:#b00020;'>Please enter a search term.</div>"

    try:
        # Assuming run_query is defined elsewhere as per your snippet
        resp = run_query(query_text) 
        rows = parse_es_results(resp)

        if not rows:
            return "<div>No results.</div>"

        cards = []
        for r in rows:
            # Prepare data for JavaScript logging
            log_data = json.dumps({
                "ecode": r['ecode'], 
                "prod_type": r['product_type'], 
                "gender": r['gender'], 
                "title": r['title']
            }).replace('"', '&quot;')

            cards.append(f"""
            <div style="
                border: 2px solid #374151; /* Darker border (Requirement 4) */
                border-radius:12px;
                padding:12px 14px;
                margin-bottom:10px;
                background:#ffffff;
                display: flex;
                justify-content: space-between;
                align-items: flex-start;
            ">
                <div style="flex-grow: 1;">
                    <div style="font-size:14px; font-weight:bold; color:#111827; margin-bottom:4px;">
                        {r.get("title")}
                    </div>
                    <div style="font-size:13px;">
                        <span style="font-weight:600; color:#6B7280;">ecode:</span>
                        <span style="font-family:monospace; color:#000; font-weight:700;">{r.get("ecode")}</span>
                        <span style="margin-left:15px; font-weight:600; color:#6B7280;">Type:</span> {r.get("product_type")}
                        <span style="margin-left:15px; font-weight:600; color:#6B7280;">Gender:</span> {r.get("gender")}
                    </div>
                    <div style="margin-top:8px; font-size:13px; color:#374151; line-height:1.35;">
                        {r.get("llm_description")}
                    </div>
                </div>
                
                <div style="margin-left: 10px;">
                    <button onclick="logDislike('{log_data}', this)" style="
                        background: #fee2e2;
                        color: #dc2626;
                        border: 1px solid #dc2626;
                        border-radius: 4px;
                        cursor: pointer;
                        padding: 2px 8px;
                        font-weight: bold;
                    ">✕</button>
                </div>
            </div>
            """)

        return f"""<div style="max-height:520px; overflow-y:auto; padding-right:6px;">
                    {''.join(cards)}
                  </div>"""

    except Exception as e:
        return f"<div style='color:#b00020;'>Error: {str(e)}</div>"

# -------------------------------------------------------------------
# Gradio UI with JavaScript Bridge
# -------------------------------------------------------------------
CSS = """
#app_title {
  font-weight: 900;
  color: #0a8f3c; 
  font-size: 38px; /* Bigger font size (Requirement 5) */
  margin: 10px 0 20px 0;
  text-align: center;
}
.hidden-component { display: none !important; }
"""

JS_LOGGING = """
function logDislike(data, btn) {
    // Fill the hidden textbox
    const textbox = document.querySelector('#log_pipe textarea');
    const event = new Event('input', { bubbles: true });
    textbox.value = data;
    textbox.dispatchEvent(event);
    
    // Trigger the hidden button
    setTimeout(() => {
        document.querySelector('#log_btn').click();
    }, 50);

    // Visual feedback: dim the card
    btn.parentElement.parentElement.style.opacity = '0.3';
    btn.disabled = true;
    btn.innerText = 'Logged';
}
"""

with gr.Blocks(css=CSS, js=JS_LOGGING) as demo:
    gr.HTML("<div id='app_title'>Dick's Vector Search Application</div>")

    # Hidden components for the bridge
    log_pipe = gr.Textbox(visible=False, elem_id="log_pipe")
    log_btn = gr.Button(visible=False, elem_id="log_btn")
    log_btn.click(fn=log_disliked_item, inputs=log_pipe)

    with gr.Row():
        query_in = gr.Textbox(
            label="Search",
            placeholder="e.g., feather coat, garden shoes...",
            lines=1
        )
        search_btn = gr.Button("Search", variant="primary")

    results_html = gr.HTML(value="<div>Enter a query and click Search.</div>")

    search_btn.click(fn=ui_search, inputs=query_in, outputs=results_html)
    query_in.submit(fn=ui_search, inputs=query_in, outputs=results_html)

demo.launch(server_name="0.0.0.0", server_port=7863)

/var/folders/36/3pkr81w50c72tdrmbywpc2540000gp/T/ipykernel_20628/1937342458.py:155: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css, js. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CSS, js=JS_LOGGING) as demo:


* Running on local URL:  http://0.0.0.0:7863
* To create a public link, set `share=True` in `launch()`.


vs_e5_llm_product_desc_embedding
size= 48
query: feather coat
vs_e5_llm_product_desc_embedding
size= 48
query: feather coat


In [17]:
import os
print(f"Your log file is located at: {os.path.join(os.getcwd(), 'disliked_records.log')}")

Your log file is located at: /Users/dks0802651/work/vector_search/notebooks/disliked_records.log
